# 03 · Análisis de métricas y comparación NLTK vs spaCy

**Etapa 4 del pipeline.**



### Preguntas de investigación elegidas (2 por eje)

| Eje | Pregunta |
|---|---|
| **A1** | ¿Las reseñas de Terror usan más adjetivos que las de Comedia? |
| **A2** | ¿Las reseñas de Acción tienen mayor proporción de verbos que las de Drama? |
| **B1** | ¿Las reseñas con calificación baja usan más negaciones que las de calificación alta? |
| **B6** | ¿Las reseñas negativas son más largas o tienen oraciones más largas que las positivas? |
| **C2** | ¿Ha cambiado la proporción de adjetivos a lo largo del tiempo? |
| **C4** | ¿Cambia con los años la longitud media de las reseñas? |



In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

# Raíz del proyecto: el notebook se ejecuta desde la carpeta notebooks/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import metrics as mt
importlib.reload(mt)

DIR_RESULTADOS = RAIZ / "data" / "results"
RUTA_POS = DIR_RESULTADOS / "pos_analysis.csv"
RUTA_LIMPIAS = RAIZ / "data" / "processed" / "resenas_limpias.csv"
RUTA_METRICAS = DIR_RESULTADOS / "metricas_por_resena.csv"
RUTA_RESPUESTAS = DIR_RESULTADOS / "respuestas_preguntas.csv"
RUTA_REVISION = DIR_RESULTADOS / "revision_manual.csv"

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.float_format", "{:.3f}".format)

# cargar_tokens evita que palabras como "NA" o "null" se lean como valores vacíos
tokens = mt.cargar_tokens(RUTA_POS)
resenas = pd.read_csv(RUTA_LIMPIAS)
resenas["valoracion"] = pd.Categorical(resenas["valoracion"], categories=["negativa", "intermedia", "positiva"], ordered=True)

print(f"{len(tokens):,} tokens de {tokens['id'].nunique()} reseñas")

639,598 tokens de 3000 reseñas


In [2]:
def mostrar(resultados):
    # Tabla de resultados de las pruebas con el valor p legible
    tabla = pd.DataFrame(resultados if isinstance(resultados, list) else [resultados])
    for columna in ("p (Mann-Whitney)", "p"):
        if columna in tabla:
            tabla[columna] = tabla[columna].map(lambda p: "< 0,001" if p < 0.001 else f"{p:.3f}")
    return tabla

## 1. Métricas por reseña

`mt.metricas_por_resena` calcula, para cada reseña, las métricas de la sección 3.4 del enunciado a partir de las etiquetas de **spaCy**. Se usa spaCy porque, además de UPOS, aporta lemas y rasgos morfológicos (como la persona de los pronombres); la sección 3 confirma además que es la herramienta más confiable para este corpus.

| Métrica | Cálculo |
|---|---|
| Densidad léxica | (NOUN + VERB + ADJ + ADV) / tokens, en % |
| Ratio verbos/sustantivos | VERB / NOUN (alto = narrativo, bajo = descriptivo) |
| Índice de verbos y de adjetivación | VERB y ADJ por cada 100 tokens |
| Índice de negación | negaciones por cada 100 tokens |
| Pronombres por persona | PRON con `Person=1`, `2` o `3` por cada 100 tokens |
| Diversidad léxica (TTR) | lemas distintos / 50, sobre los **primeros 50 tokens** de cada reseña |
| Longitud media de oración | tokens / oraciones (segmentación de spaCy) |


In [4]:
metricas = mt.metricas_por_resena(tokens, tamano_ttr=50)
df = resenas[["id", "genero", "periodo", "anio", "calificacion", "valoracion", "n_palabras"]].merge(metricas, on="id")

df.to_csv(RUTA_METRICAS, index=False, encoding="utf-8")
print(f"Guardado {RUTA_METRICAS.relative_to(RAIZ)}: {len(df)} reseñas")
print(f"Reseñas sin TTR (menos de 50 tokens): {df['ttr'].isna().sum()}")
df[list(mt.NOMBRES_METRICAS)].describe().T.rename(index=mt.NOMBRES_METRICAS)

Guardado data\results\metricas_por_resena.csv: 3000 reseñas
Reseñas sin TTR (menos de 50 tokens): 51


,count,mean,std,min,25%,50%,75%,max
Tokens (sin puntuación),3000.000,185.215,88.003,30.000,125.000,163.000,236.000,451.000
Densidad léxica (%),3000.000,44.910,3.985,29.762,42.328,45.000,47.500,59.259
Ratio verbos/sustantivos,3000.000,0.598,0.238,0.077,0.435,0.561,0.714,3.000
Verbos por 100 tokens,3000.000,10.607,2.965,1.786,8.665,10.417,12.500,26.415
Adjetivos por 100 tokens,3000.000,8.538,2.709,0.000,6.723,8.333,10.169,22.222
Negaciones por 100 tokens,3000.000,1.987,1.454,0.000,0.935,1.765,2.778,10.811
Pronombres 1.ª persona por 100 tokens,3000.000,0.858,1.002,0.000,0.000,0.630,1.325,10.280
Pronombres 2.ª persona por 100 tokens,3000.000,0.339,0.656,0.000,0.000,0.000,0.526,5.714
Pronombres 3.ª persona por 100 tokens,3000.000,2.301,1.387,0.000,1.357,2.190,3.117,9.524
"Diversidad léxica (TTR, primeros 50 tokens)",2949.000,0.744,0.062,0.340,0.700,0.740,0.780,1.000


### 1.1 Promedios por grupo

Panorama de todas las métricas por género, valoración y periodo. Las preguntas de la sección 2 profundizan en algunas de ellas.

In [5]:
mt.resumen_por_grupo(df, "genero").rename(columns=mt.NOMBRES_METRICAS)

,n,Tokens (sin puntuación),Densidad léxica (%),Ratio verbos/sustantivos,Verbos por 100 tokens,Adjetivos por 100 tokens,Negaciones por 100 tokens,Pronombres 1.ª persona por 100 tokens,Pronombres 2.ª persona por 100 tokens,Pronombres 3.ª persona por 100 tokens,"Diversidad léxica (TTR, primeros 50 tokens)",Longitud media de oración (tokens)
genero,,,,,,,,,,,,
Action,600,193.817,44.456,0.572,10.132,8.650,1.975,0.742,0.293,2.216,0.747,19.003
Animation,600,187.230,44.806,0.579,10.350,8.763,1.804,0.950,0.313,2.175,0.742,20.471
Comedy,600,175.382,44.449,0.615,10.615,8.581,2.010,0.835,0.370,2.440,0.744,20.190
Drama,600,186.212,45.131,0.597,10.731,8.441,1.966,0.868,0.321,2.337,0.743,21.267
Horror,600,183.435,45.711,0.625,11.206,8.254,2.181,0.894,0.400,2.337,0.742,20.129


In [6]:
mt.resumen_por_grupo(df, "valoracion").rename(columns=mt.NOMBRES_METRICAS)

,n,Tokens (sin puntuación),Densidad léxica (%),Ratio verbos/sustantivos,Verbos por 100 tokens,Adjetivos por 100 tokens,Negaciones por 100 tokens,Pronombres 1.ª persona por 100 tokens,Pronombres 2.ª persona por 100 tokens,Pronombres 3.ª persona por 100 tokens,"Diversidad léxica (TTR, primeros 50 tokens)",Longitud media de oración (tokens)
valoracion,,,,,,,,,,,,
negativa,684,181.618,45.992,0.629,11.240,8.472,2.617,0.835,0.263,2.348,0.748,19.891
intermedia,374,197.896,45.035,0.582,10.348,8.362,2.338,0.827,0.203,2.389,0.745,20.471
positiva,1942,184.040,44.506,0.589,10.434,8.594,1.698,0.872,0.393,2.268,0.742,20.275


In [7]:
mt.resumen_por_grupo(df, "periodo").rename(columns=mt.NOMBRES_METRICAS)

,n,Tokens (sin puntuación),Densidad léxica (%),Ratio verbos/sustantivos,Verbos por 100 tokens,Adjetivos por 100 tokens,Negaciones por 100 tokens,Pronombres 1.ª persona por 100 tokens,Pronombres 2.ª persona por 100 tokens,Pronombres 3.ª persona por 100 tokens,"Diversidad léxica (TTR, primeros 50 tokens)",Longitud media de oración (tokens)
periodo,,,,,,,,,,,,
1998-2009,1500,173.906,44.715,0.607,10.652,8.495,1.982,0.853,0.351,2.258,0.745,19.266
2010-2018,1500,196.524,45.106,0.588,10.562,8.580,1.993,0.863,0.328,2.344,0.743,21.158


## 2. Preguntas de investigación



### Eje A · Comparación entre géneros

#### A1. ¿Las reseñas de Terror usan más adjetivos que las de Comedia?
Métrica: **adjetivos por cada 100 tokens**

In [8]:
display(mostrar(mt.comparar_grupos(df, "idx_adjetivos", "genero", "Horror", "Comedy")))
df.groupby("genero")["idx_adjetivos"].agg(["mean", "median", "std", "size"]).sort_values("mean", ascending=False)

,métrica,grupo A,media A,mediana A,n A,grupo B,media B,mediana B,n B,diferencia (A − B),p (Mann-Whitney),efecto r,tamaño del efecto
0,Adjetivos por 100 tokens,Horror,8.254,8.029,600,Comedy,8.581,8.333,600,-0.327,0.067,-0.061,despreciable


,mean,median,std,size
genero,,,,
Animation,8.763,8.527,2.716,600
Action,8.650,8.424,2.679,600
Comedy,8.581,8.333,2.759,600
Drama,8.441,8.264,2.677,600
Horror,8.254,8.029,2.694,600


**Interpretación A1.** **No.** Las reseñas de Terror no usan más adjetivos que las de Comedia. Tienen en promedio 8,25 adjetivos por cada 100 tokens, frente a 8,58 en Comedia, la diferencia no es significativa  Terror es incluso el género con el índice de adjetivación más bajo de los cinco, aunque las diferencias entre géneros son de décimas sobre una media cercana a 8,5.


#### A2. ¿Las reseñas de Acción tienen mayor proporción de verbos que las de Drama?
Métrica principal: **verbos por cada 100 tokens**. Se muestra también el **ratio verbos/sustantivos**, que indica estilo narrativo (alto) frente a descriptivo (bajo).

In [8]:
display(mostrar(mt.comparar_grupos(df, "idx_verbos", "genero", "Action", "Drama")))
df.groupby("genero")[["idx_verbos", "ratio_verbos_sustantivos"]].mean().sort_values("idx_verbos", ascending=False)

,métrica,grupo A,media A,mediana A,n A,grupo B,media B,mediana B,n B,diferencia (A − B),p (Mann-Whitney),efecto r,tamaño del efecto
0,Verbos por 100 tokens,Action,10.132,9.898,600,Drama,10.731,10.448,600,-0.599,0.001,-0.110,pequeño


,idx_verbos,ratio_verbos_sustantivos
genero,,
Horror,11.206,0.625
Drama,10.731,0.597
Comedy,10.615,0.615
Animation,10.350,0.579
Action,10.132,0.572


**Interpretación A2.** **No; ocurre lo contrario.** Las reseñas de Acción tienen *menos* verbos que las de Drama: 10,13 frente a 10,73 por cada 100 tokens. Acción es el género con menos verbos y el de ratio verbos/sustantivos más bajo, y Terror el de más verbos. La hipótesis "más acción en pantalla → más verbos en la reseña" no se cumple: las reseñas de Acción parecen más descriptivas (efectos, escenas, actores) que narrativas.


### Eje B · Emocionalidad gramatical

#### B1. ¿Las reseñas con calificación baja usan más negaciones que las de calificación alta?
Métrica: **negaciones por cada 100 tokens**. Se compara la valoración negativa (1–4) con la positiva (7–10), y se revisa toda la escala de 1 a 10.

In [9]:
display(mostrar(mt.comparar_grupos(df, "idx_negacion", "valoracion", "negativa", "positiva")))
df.groupby("calificacion")["idx_negacion"].agg(["mean", "median", "size"]).T

,métrica,grupo A,media A,mediana A,n A,grupo B,media B,mediana B,n B,diferencia (A − B),p (Mann-Whitney),efecto r,tamaño del efecto
0,Negaciones por 100 tokens,negativa,2.617,2.448,684,positiva,1.698,1.531,1942,0.919,"< 0,001",0.358,mediano


calificacion,1,2,3,4,5,6,7,8,9,10
mean,2.767,2.261,2.569,2.610,2.591,2.143,1.840,1.657,1.610,1.704
median,2.564,2.000,2.650,2.326,2.339,1.852,1.617,1.550,1.394,1.538
size,318.000,115.000,130.000,121.000,163.000,211.000,346.000,441.000,402.000,753.000


**Interpretación B1.** **Sí, y es el resultado más claro de todo el análisis.** Las reseñas negativas usan unas 2,6 negaciones por cada 100 tokens, frente a 1,7 en las positivas

quien critica una obra rechaza o contradice ("no funciona", "nunca llega", "nada original").

#### B6. ¿Las reseñas negativas son más largas o tienen oraciones más largas que las positivas?
Métricas: **longitud de la reseña** (tokens sin puntuación) y **longitud media de oración** (tokens por oración).

In [11]:
mostrar([mt.comparar_grupos(df, "n_tokens", "valoracion", "negativa", "positiva"),
         mt.comparar_grupos(df, "long_oracion", "valoracion", "negativa", "positiva")])

,métrica,grupo A,media A,mediana A,n A,grupo B,media B,mediana B,n B,diferencia (A − B),p (Mann-Whitney),efecto r,tamaño del efecto
0,Tokens (sin puntuación),negativa,181.618,161.000,684,positiva,184.040,161.000,1942,-2.421,0.999,0.000,despreciable
1,Longitud media de oración (tokens),negativa,19.891,17.829,684,positiva,20.275,18.000,1942,-0.384,0.810,-0.006,despreciable


In [12]:
df.groupby("valoracion", observed=True)[["n_tokens", "long_oracion"]].agg(["mean", "median"])

n_tokens         long_oracion       
               mean  median         mean median
valoracion                                     
negativa    181.618 161.000       19.891 17.829
intermedia  197.896 182.000       20.471 18.481
positiva    184.040 161.000       20.275 18.000

**Interpretación B6.** **No.** Las reseñas negativas y las positivas tienen prácticamente la misma longitud (unos 182 frente a 184 tokens de media, con la misma mediana de 161) y oraciones de longitud equivalente (19,9 frente a 20,3 tokens).

### Eje C · Evolución temporal

#### C2. ¿Ha cambiado la proporción de adjetivos a lo largo del tiempo?
Métrica: **adjetivos por cada 100 tokens**.

In [13]:
mostrar([mt.comparar_grupos(df, "idx_adjetivos", "periodo", "2010-2018", "1998-2009")])

,métrica,grupo A,media A,mediana A,n A,grupo B,media B,mediana B,n B,diferencia (A − B),p (Mann-Whitney),efecto r,tamaño del efecto
0,Adjetivos por 100 tokens,2010-2018,8.580,8.351,1500,1998-2009,8.495,8.333,1500,0.085,0.311,0.021,despreciable


In [14]:
display(mostrar(mt.tendencia(df, "idx_adjetivos")))
df.groupby("anio")["idx_adjetivos"].agg(["mean", "size"]).T

,métrica,rho de Spearman,p,n
0,Adjetivos por 100 tokens,0.018,0.336,3000


anio,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,...,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
mean,8.007,8.731,8.680,8.459,8.742,8.583,8.936,8.429,8.352,8.692,...,8.287,8.511,8.666,8.215,8.515,8.623,8.447,8.617,8.853,7.883
size,24.000,100.000,110.000,125.000,104.000,130.000,88.000,166.000,198.000,152.000,...,145.000,154.000,152.000,142.000,198.000,190.000,196.000,190.000,275.000,3.000


**Interpretación C2.** **No.** La proporción de adjetivos es estable: 8,50 por cada 100 tokens en 1998–2009 y 8,58 en 2010–2018.

#### C4. ¿Cambia con los años la longitud media de las reseñas?
Métrica: **tokens por reseña** (sin puntuación). Como comprobación, se repite con el número de palabras del **texto original en inglés** (`n_palabras`), para descartar que el cambio lo haya introducido la traducción.

In [15]:
mostrar([mt.comparar_grupos(df, "n_tokens", "periodo", "2010-2018", "1998-2009"),
         mt.comparar_grupos(df, "n_palabras", "periodo", "2010-2018", "1998-2009")])

,métrica,grupo A,media A,mediana A,n A,grupo B,media B,mediana B,n B,diferencia (A − B),p (Mann-Whitney),efecto r,tamaño del efecto
0,Tokens (sin puntuación),2010-2018,196.524,174.000,1500,1998-2009,173.906,152.500,1500,22.618,"< 0,001",0.167,pequeño
1,n_palabras,2010-2018,189.352,167.000,1500,1998-2009,168.353,150.000,1500,20.999,"< 0,001",0.163,pequeño


In [16]:
display(mostrar([mt.tendencia(df, "n_tokens"), mt.tendencia(df, "n_palabras")]))
df.groupby("anio")["n_tokens"].agg(["mean", "median", "size"]).T

,métrica,rho de Spearman,p,n
0,Tokens (sin puntuación),0.189,"< 0,001",3000
1,n_palabras,0.185,"< 0,001",3000


anio,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,...,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
mean,129.625,122.560,129.664,136.520,135.769,151.031,175.170,189.066,191.348,196.697,...,223.924,207.669,207.974,207.754,199.561,199.037,203.745,197.053,168.498,217.000
median,113.000,108.000,104.500,119.000,107.000,122.000,150.000,163.500,164.000,165.000,...,190.000,183.500,182.000,197.000,175.500,175.000,179.500,178.000,141.000,229.000
size,24.000,100.000,110.000,125.000,104.000,130.000,88.000,166.000,198.000,152.000,...,145.000,154.000,152.000,142.000,198.000,190.000,196.000,190.000,275.000,3.000


**Interpretación C4.** **Sí.** Las reseñas recientes son más largas: 196,5 tokens de media en 2010–2018 frente a 173,9 en 1998–2009, unos 23 tokens más (+13 %)

## 3. Comparación NLTK vs spaCy

Ambas herramientas etiquetaron **los mismos tokens** (la tokenización de spaCy), así que la comparación es directa, token a token

### 3.1 Concordancia global

In [18]:
print(f"Concordancia global NLTK–spaCy (sin puntuación): {mt.concordancia_global(tokens):.1f} %")

Concordancia global NLTK–spaCy (sin puntuación): 82.8 %


### 3.2 Concordancia por categoría

Se mide desde los dos lados: *según spaCy* indica, de los tokens que spaCy etiqueta como X, cuántos etiqueta NLTK también como X; *según NLTK* indica lo mismo, partiendo de las etiquetas de NLTK.

In [19]:
mt.concordancia_por_categoria(tokens)

,tokens spaCy,% coincide (según spaCy),tokens NLTK,% coincide (según NLTK)
NOUN,103669,95.274,162688,60.711
DET,78944,98.058,86793,89.190
ADP,74647,99.859,74850,99.589
VERB,58333,70.375,51318,79.995
ADJ,46901,66.201,34376,90.322
PRON,40277,80.257,36898,87.606
ADV,39349,93.184,38160,96.088
AUX,33641,71.333,24195,99.182
PROPN,32823,8.750,3697,77.685
CCONJ,22905,99.035,22726,99.815


**Interpretación:**
- **Coinciden casi siempre** en las categorías funcionales cerradas, que son listas pequeñas de palabras muy frecuentes y bien representadas: preposiciones (ADP), conjunciones (CCONJ), determinantes (DET) y adverbios (ADV), todas por encima del 93 % según spaCy.
- **Coinciden menos** en las categorías abiertas (VERB, ADJ) y en AUX, y **muy poco en PROPN**. NLTK casi nunca etiqueta nombres propios
- NLTK etiqueta como NOUN muchos más tokens que spaCy. Es el efecto del respaldo `DefaultTagger("NOUN")`: toda palabra desconocida acaba siendo sustantivo

### 3.3 Matriz de confusión

Filas = etiqueta de spaCy; columnas = etiqueta de NLTK. Cada fila suma 100 %: muestra cómo etiquetó NLTK los tokens de cada categoría de spaCy

In [20]:
matriz = mt.matriz_confusion(tokens)
matriz.style.background_gradient(cmap="Blues", axis=None).format("{:.1f}")

upos_nltk,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PRON,PROPN,PUNCT,SCONJ,VERB,X
upos_spacy,,,,,,,,,,,,,,,
ADJ,66.2,0.0,0.5,0.0,0.0,0.7,0.1,30.3,0.0,0.0,0.2,0.0,0.0,1.9,0.0
ADP,0.0,99.9,0.0,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ADV,0.4,0.0,93.2,0.1,0.1,0.2,0.0,4.2,0.0,1.5,0.0,0.0,0.1,0.2,0.0
AUX,0.0,0.0,0.2,71.3,0.0,0.0,0.0,3.9,0.0,0.0,0.2,0.0,0.0,24.4,0.0
CCONJ,0.0,0.0,0.0,0.0,99.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.8,0.0,0.0
DET,0.0,0.3,0.2,0.0,0.0,98.1,0.0,0.1,0.0,1.3,0.0,0.0,0.0,0.0,0.0
INTJ,20.6,0.0,30.5,0.3,0.0,0.0,17.4,25.7,0.0,0.5,1.9,0.0,0.0,3.1,0.0
NOUN,1.8,0.0,0.4,0.1,0.0,0.5,0.0,95.3,0.4,0.1,0.6,0.0,0.0,0.8,0.0
NUM,0.1,0.0,0.0,0.0,0.0,26.9,0.0,14.5,54.0,4.3,0.0,0.1,0.0,0.0,0.0


**Interpretación de la matriz:**
- **Columna NOUN:** es el principal destino de las discrepancias. Allí van la mayoría de los PROPN de spaCy y cerca de un 30 % de sus VERB y ADJ. Son palabras que NLTK no conoce y a las que asigna la etiqueta por defecto.
- **AUX → VERB** (24 % de los AUX de spaCy): no es un error, sino un **criterio distinto**. spaCy trata *estar*, *poder*, *ir* o *deber* como auxiliares; `cess_esp` los anota como verbos principales.
- **PRON → DET** (18 %): palabras como *esta*, *una*, *todo* o *lo*, que pueden ser determinante ("esta película") o pronombre ("esta es buena"). Es **ambigüedad real**: el bigrama de NLTK no siempre la resuelve.


### 3.4 Discrepancias y su causa probable

`mt.causa_probable` pre-clasifica automáticamente cada discrepancia:

1. **Palabra ausente del entrenamiento de NLTK:** la palabra no aparece en `cess_esp` (columna `en_cess`).
2. **Diferencia de criterio entre esquemas:** pares AUX↔VERB, SCONJ↔PRON y NUM→DET.
3. **Ambigüedad real:** el resto. Palabras conocidas cuya categoría depende del contexto.

La cuarta causa del enunciado, el **error de traducción**, no se puede detectar automáticamente. Se identifica en la revisión manual (sección 3.6), comparando con el texto original en inglés.

In [21]:
disc = mt.discrepancias(tokens)
print(f"Tokens en discrepancia: {len(disc):,} ({len(disc) / len(mt.sin_puntuacion(tokens)) * 100:.1f} % de los tokens sin puntuación)\n")
display(disc["causa_probable"].value_counts().to_frame("tokens").assign(**{"%": lambda t: t["tokens"] / t["tokens"].sum() * 100}))
disc.groupby(["upos_spacy", "upos_nltk"]).size().sort_values(ascending=False).head(10).to_frame("tokens")

Tokens en discrepancia: 95,471 (17.2 % de los tokens sin puntuación)



,tokens,%
causa_probable,,
Palabra ausente del entrenamiento de NLTK,59090,61.893
Ambigüedad real,23984,25.122
Diferencia de criterio entre esquemas,12397,12.985


,,tokens
upos_spacy,upos_nltk,
PROPN,NOUN,28805
VERB,NOUN,16774
ADJ,NOUN,14202
AUX,VERB,8213
PRON,DET,7247
SCONJ,PRON,2653
NOUN,ADJ,1830
ADV,NOUN,1637
AUX,NOUN,1304


### 3.5 Diez ejemplos de discrepancia, por causa

Se toman al azar (semilla fija, para que sean reproducibles) 4 ejemplos de palabras ausentes, 3 de diferencia de criterio y 3 de ambigüedad, con la oración completa como contexto.

In [22]:
cuantos = {"Palabra ausente del entrenamiento de NLTK": 4,
           "Diferencia de criterio entre esquemas": 3,
           "Ambigüedad real": 3}
ejemplos = pd.concat([disc[disc["causa_probable"] == causa].sample(n, random_state=42)
                      for causa, n in cuantos.items()])
ejemplos[["causa_probable", "id", "token", "upos_spacy", "upos_nltk", "contexto"]].reset_index(drop=True)

,causa_probable,id,token,upos_spacy,upos_nltk,contexto
0,Palabra ausente del entrenamiento de NLTK,2075,desenfrenado,ADJ,NOUN,"Forma amistades con un hombre desenfrenado inseguro , interpretado por John C. Reilly cuyo castigo por masturbarse es quemarse la mano en una tostadora ."
1,Palabra ausente del entrenamiento de NLTK,651,Amanda,PROPN,NOUN,"Iris se enamora de la amiga de Amanda , Miles ( Negro ) y se hace amiga de su vecino , un veterano de cine , Arthur Abbott ( Wallach ) ."
2,Palabra ausente del entrenamiento de NLTK,1893,Eva,PROPN,NOUN,Es casi como si los cambios fueran fragmentos de los recuerdos de Eva mientras camina a través de su vida actual tratando de unir lo que salió mal .
3,Palabra ausente del entrenamiento de NLTK,1964,Mimic,PROPN,NOUN,"Mimic es una película de Monsert , y justo eso ."
4,Diferencia de criterio entre esquemas,550,están,AUX,VERB,Creo que los adultos están sobreestimando lo que los niños no pueden manejar .
5,Diferencia de criterio entre esquemas,269,están,AUX,VERB,Los personajes principales están muy bien desarrollados y todos tienen su propia personalidad .
6,Diferencia de criterio entre esquemas,1089,que,SCONJ,PRON,"Puede haber sido mi imaginación , pero en realidad se puede ver la creciente conciencia de la difícil situación de los actores amaneciendo en ellos a medida que la película progresa , a medida que..."
7,Ambigüedad real,724,esta,PRON,DET,"En resumen , no , esta no es la mejor película que jamás hayas visto ."
8,Ambigüedad real,2197,algunas,PRON,DET,Como algunas de las peores películas de Spider Man .
9,Ambigüedad real,2215,lo,PRON,DET,"Sin embargo , cuando ella a regañadientes está de acuerdo y lo besa , ella misma se transforma en una rana ."


### 3.6 Revisión manual de 50 tokens en discrepancia

El enunciado pide respaldar la conclusión con la **revisión manual de al menos 50 tokens** en discrepancia.

**Cómo hacerla:**
1. La celda siguiente crea `data/results/revision_manual.csv` con 50 discrepancias elegidas al azar (semilla fija). Si el archivo ya existe **no lo sobrescribe**, para no perder su trabajo.
2. Ábranlo en Excel (o en otro editor) y, para cada fila, completen:
   - `correcta`: qué herramienta acertó, **spaCy**, **NLTK** o **ninguna**.
   - `causa_revisada`: **ausente**, **criterio**, **ambigüedad** o **traducción**. Esta última se usa cuando el token es raro porque la traducción es mala; para comprobarlo se compara `contexto` con `texto_en`.
   - `comentario` (opcional): por qué.
3. Guarden el archivo **en formato CSV UTF-8** y ejecuten la celda de resumen

In [3]:
if RUTA_REVISION.exists():
    print(f"{RUTA_REVISION.relative_to(RAIZ)} ya existe: no se sobrescribe.")
else:
    muestra = disc.sample(50, random_state=42).merge(resenas[["id", "texto_en"]], on="id")
    plantilla = muestra[["id", "oracion", "token", "lema", "upos_spacy", "upos_nltk", "en_cess",
                         "causa_probable", "contexto", "texto_en"]].copy()
    plantilla["correcta"] = ""
    plantilla["causa_revisada"] = ""
    plantilla["comentario"] = ""
    plantilla.to_csv(RUTA_REVISION, index=False, encoding="utf-8-sig")  # utf-8-sig: Excel muestra bien las tildes
    print(f"Creada {RUTA_REVISION.relative_to(RAIZ)} con {len(plantilla)} tokens para revisar.")

data\results\revision_manual.csv ya existe: no se sobrescribe.


### 3.7 Resultado de la revisión manual

In [4]:
revision = pd.read_csv(RUTA_REVISION, keep_default_na=False)
completas = revision[revision["correcta"].str.strip() != ""]

if len(completas) < len(revision):
    print(f"Revisión incompleta: {len(completas)} de {len(revision)} tokens revisados. "
          "Completen las columnas 'correcta' y 'causa_revisada' y vuelvan a ejecutar esta celda.")
else:
    print("¿Quién acertó? (% de los 50 tokens)")
    display((completas["correcta"].str.strip().str.lower().value_counts(normalize=True) * 100).round(1).to_frame("%"))
    print("Causa según la revisión manual:")
    display(completas["causa_revisada"].str.strip().str.lower().value_counts().to_frame("tokens"))
    print("Pre-clasificación automática vs revisión manual:")
    display(pd.crosstab(completas["causa_probable"], completas["causa_revisada"].str.strip().str.lower()))

¿Quién acertó? (% de los 50 tokens)


,%
correcta,
spacy,88.000
nltk,12.000


Causa según la revisión manual:


,tokens
causa_revisada,
ausente,28
ambigüedad,15
criterio,6
traducción,1


Pre-clasificación automática vs revisión manual:


causa_revisada,ambigüedad,ausente,criterio,traducción
causa_probable,,,,
Ambigüedad real,9,0,0,0
Diferencia de criterio entre esquemas,6,0,6,0
Palabra ausente del entrenamiento de NLTK,0,28,0,1


### 3.8 Conclusión: ¿qué herramienta es más confiable para este corpus?

*Borrador basado en el análisis automático. Completen los datos entre corchetes con el resultado de la revisión manual (3.7) y ajusten el texto si no coincide.*

Las dos herramientas coinciden en cerca del 83 % de los tokens. El análisis de las discrepancias indica que **spaCy es la herramienta más confiable para este corpus**:

- **Más de la mitad de las discrepancias (≈ 62 %) son palabras que NLTK no conoce.** El etiquetador entrenado con `cess_esp` (unas 6.000 oraciones de prensa) no tiene vocabulario de cine ni nombres propios de actores o películas, y les asigna NOUN por defecto. spaCy, entrenado con un corpus mucho mayor y con representaciones de palabras que generalizan a palabras no vistas, no tiene ese problema.
- **Cerca de un 13 % son diferencias de criterio** (AUX/VERB, SCONJ/PRON y NUM/DET). No son errores de ninguna herramienta, pero hay que tenerlas en cuenta al comparar distribuciones.
- **El resto (≈ 25 %) es ambigüedad real**, donde el contexto de una sola palabra anterior que usa el bigrama de NLTK se queda corto frente al modelo contextual de spaCy.
- En la revisión manual de 50 tokens, spaCy acertó en **88 %** de los casos, NLTK en **12 %** . **1** caso se a error de traducción.